# 05 - External Legacy Fortran Oracle

The Python runtime does not call Fortran. Fortran remains an optional numerical oracle
used to explain and reproduce compatibility decisions. Set `BESTPRED_FORTRAN_BINARY` to
a compatible executable, for example:

```bash
export BESTPRED_FORTRAN_BINARY=/path/to/standalone-bestpred/bestpred
```

The known sibling standalone checkout is detected as a convenience. No executable is
committed to Bovi. The available historical binary is Linux x86-64 and requires
`libgfortran.so.5`; other platforms must build or supply their own binary.

In [1]:
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "packages/models/bestpred").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from a Bovi repository checkout")


ROOT = find_repo_root()
PACKAGE_ROOT = ROOT / "packages/models/bestpred"
FIXTURES = PACKAGE_ROOT / "tests/fixtures"
PARAMETERS = FIXTURES / "source11_current/bestpred.par"

In [2]:
import os
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd

from bestpred.core.kernel import predict_records
from bestpred.core.source11 import simulate_source11_records
from bestpred.io.dcr import read_dcr_results
from bestpred.io.parameters import read_parameters
from bestpred.io.source11 import read_source11_examples


def discover_fortran_binary() -> Path | None:
    if os.getenv("BESTPRED_DISABLE_FORTRAN") == "1":
        return None
    configured = os.getenv("BESTPRED_FORTRAN_BINARY")
    if configured:
        candidate = Path(configured).expanduser()
        return candidate if candidate.is_file() else None
    for parent in (ROOT, *ROOT.parents):
        candidate = parent / "bestpred/bestpred"
        if candidate.is_file():
            return candidate
    return None


fortran_binary = discover_fortran_binary()
print(
    "Oracle mode:",
    "external Fortran executable" if fortran_binary else "checked-in current-Fortran fixture",
)

Oracle mode: external Fortran executable


## Run Fortran when available, otherwise use the golden output

The source-11 Fortran reader flushes the final testing plan only after a blank separator. The temporary copy therefore receives an explicit trailing blank line; the repository fixture itself remains normalized.

In [3]:
source11 = FIXTURES / "source11_current"
oracle_rows = None
fortran_note = "Golden fixture loaded; no external executable was available."

if fortran_binary is not None:
    try:
        with tempfile.TemporaryDirectory(prefix="bestpred-fortran-") as temp_dir:
            work = Path(temp_dir)
            executable = work / "bestpred"
            shutil.copy2(fortran_binary, executable)
            executable.chmod(0o755)
            shutil.copy2(source11 / "bestpred.par", work / "bestpred.par")
            input_path = work / "DCRexample.txt"
            shutil.copy2(source11 / "DCRexample.txt", input_path)
            input_path.write_text(input_path.read_text().rstrip() + "\n\n")
            shutil.copy2(source11 / "adjust.scs", work / "adjust.scs")
            subprocess.run(
                [str(executable)],
                cwd=work,
                check=True,
                capture_output=True,
                text=True,
                timeout=120,
            )
            oracle_rows = read_dcr_results(work / "results_v2.dcr")
        fortran_note = "External Fortran run completed in an isolated temporary directory."
    except (OSError, subprocess.SubprocessError, FileNotFoundError) as exc:
        fortran_note = (
            f"External Fortran was unavailable at runtime ({type(exc).__name__}); "
            "golden fixture used."
        )

if oracle_rows is None:
    oracle_rows = read_dcr_results(source11 / "results_v2.dcr")

print(fortran_note)
print("Oracle rows:", len(oracle_rows))

External Fortran run completed in an isolated temporary directory.
Oracle rows: 43


## Run the equivalent pure-Python path

In [4]:
parameters = read_parameters(source11 / "bestpred.par")
examples = read_source11_examples(source11 / "DCRexample.txt")
records = simulate_source11_records(examples, parameters)
python_rows = predict_records(records, parameters)

numeric_equal = [
    np.allclose(
        python_row.numeric_values,
        oracle_row.numeric_values,
        rtol=0,
        atol=0.51,
        equal_nan=True,
    )
    for python_row, oracle_row in zip(python_rows, oracle_rows, strict=True)
]
comparison = pd.DataFrame(
    {
        "Rows": [len(python_rows)],
        "Rows within output-rounding tolerance": [sum(numeric_equal)],
        "Mismatched rows": [len(numeric_equal) - sum(numeric_equal)],
        "Fields per row": [len(python_rows[0].numeric_values)],
    }
)
comparison

,Rows,Rows within output-rounding tolerance,Mismatched rows,Fields per row
0,43,43,0,43


## Current Fortran is not the same as the old manual output

In [5]:
manual_rows = read_dcr_results(FIXTURES / "legacy_manual_expected/DCRexample.results.dcr")
first_current = np.asarray(oracle_rows[0].numeric_values)
first_manual = np.asarray(manual_rows[0].numeric_values)
pd.DataFrame(
    {
        "Reference": ["Current Fortran fixture", "Legacy manual distribution"],
        "Rows": [len(oracle_rows), len(manual_rows)],
        "First-row milk 305": [first_current[3], first_manual[3]],
        "First-row DCR milk": [first_current[0], first_manual[0]],
    }
)

,Reference,Rows,First-row milk 305,First-row DCR milk
0,Current Fortran fixture,43,21900.0,102.0
1,Legacy manual distribution,43,21753.0,102.0


Golden tests target the **current Fortran source behavior**. The manual-distribution
file is retained to document historical drift, not as the active oracle. Compatibility
also preserves known stateful and EOF artifacts; consult the Fortran quirks document
before treating every reproduced behavior as desirable future Python behavior.